# Evaluate the Badini CosyVoice3 model on the whole test set
Run with the **cosyvoice** kernel, after training (needs `model_ft` from section 12 of the training notebook).
Synthesizes **every** test sentence, then scores: SIM-o (prompt + real sentence), UTMOS, RTF, duration ratio, optional CER/WER.
Resumable: existing wavs are not re-synthesized.

In [1]:
import sys, subprocess, importlib.util, os
print("python:", sys.executable)          # want /workspace/envs/cosyvoice/bin/python
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
need = {"soundfile": "soundfile", "pandas": "pandas", "speechbrain": "speechbrain"}
missing = [pip for mod, pip in need.items() if importlib.util.find_spec(mod) is None]
print("missing:", missing or "nothing")
if missing:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-input", "--default-timeout", "30", *missing],
                       text=True, timeout=600)
    print("pip exit code:", r.returncode)
print("deps OK")

python: /workspace/envs/cosyvoice/bin/python
missing: nothing
deps OK


In [ ]:
import sys, subprocess, os
print('python:', sys.executable)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'soundfile', 'pandas', 'speechbrain'], check=True)

In [2]:
import os, re, sys, csv, json, time, collections
import torch, torchaudio, numpy as np, soundfile as sf

# ---- paths (same as training notebook cell 1)
DATA_ROOT = "/workspace/Data24_09_2026"
AUDIO_DIR = f"{DATA_ROOT}/clips"
TRAIN_TSV = f"{DATA_ROOT}/train_all.tsv"
TEST_TSV  = f"{DATA_ROOT}/test_all.tsv"      # the test set
SPK_COL, PATH_COL, TEXT_COL, HAS_HEADER = 0, 1, 3, True
WORK      = "/workspace/cosyvoice3_badini_all"
COSY_REPO = "/workspace/CosyVoice"
MODEL_DIR = f"{WORK}/model_ft"
INSTRUCT  = "You are a helpful assistant.<|endofprompt|>"

LIMIT     = 0        # max sentences per speaker; 0 = ALL.  Use e.g. 20 for a quick check
ASR_MODEL = ""       # optional HF id of an ASR model that handles Badini -> adds CER/WER
ASR_LANG  = ""
OUT_DIR   = f"{WORK}/eval_test_all"
SYNTH_DIR = f"{OUT_DIR}/wavs"
os.makedirs(SYNTH_DIR, exist_ok=True)
sys.path.append(f"{COSY_REPO}/third_party/Matcha-TTS"); sys.path.append(COSY_REPO)
dev = "cuda" if torch.cuda.is_available() else "cpu"

## 1 · Badini number → words converter (same as training notebook)

In [3]:
# ---- digits -> Badini words (spelling from your corrected Excel sheet; EDIT here if a word is wrong) ----
ONES  = {1:"ئێک",2:"دو",3:"سێ",4:"چار",5:"پێنج",6:"شەش",7:"حەفت",8:"هەشت",9:"نەه",
         10:"دەه",11:"یازدە",12:"دوازدە",13:"سیزدە",14:"چاردە",15:"پازدە",
         16:"شازدە",17:"حەڤدە",18:"هەژدە",19:"نوزدە"}
TENS  = {2:"بیست",3:"سیه",4:"چل",5:"پێنجی",6:"شێست",7:"حەفتێ",8:"هەشتا",9:"نوت"}
HUND, THOU, ZERO, AND = "سەد", "هزار", "سفر", " و "
POINT, HOUR, MINUTE, SECOND = "پوینت", "دەمژمێر", "خولەک", "چرکە"   # HOUR is said BEFORE the hour number
PLUS, MINUS = "کومکرنی", "ژێبرن"
DAY, MONTH, YEAR = "رۆژا", "هەیڤا", "سالا"
CURR = {"$": ("دولار", "سەنت"), "€": ("یۆرۆ", "سەنت")}   # symbol -> (main unit, cents)
D = "0-9٠-٩۰-۹"

def _below100(r):
    if r < 20: return ONES[r]
    t, u = divmod(r, 10)
    return TENS[t] + (AND + ONES[u] if u else "")

def _below1000(n):
    h, r = divmod(n, 100)
    parts = []
    if h: parts.append(HUND if h == 1 else ONES[h] + " " + HUND)
    if r: parts.append(_below100(r))
    return AND.join(parts)

def num2words_badini(n):
    if n == 0: return ZERO
    th, r = divmod(n, 1000)
    parts = []
    if th: parts.append(THOU if th == 1 else _below1000(th) + " " + THOU)
    if r:  parts.append(_below1000(r))
    return AND.join(parts)

_ASCII = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")

def _int(s):                       # digit string -> words (0-999999; longer numbers / 007 codes stay as digits)
    s = s.translate(_ASCII)
    if len(s) > 1 and s[0] == "0": return s
    n = int(s)
    return num2words_badini(n) if n <= 999999 else s

def _frac(s):                      # fraction read as a number:  23 -> بیست و سێ ;  05 -> سفر پێنج
    s = s.translate(_ASCII); z = len(s) - len(s.lstrip("0")); rest = s.lstrip("0")
    return " ".join([ZERO] * z + ([_int(rest)] if rest else []))

def _num(s):                       # 34.567 / 45.6 / 12
    s = s.translate(_ASCII)
    if "." in s:
        i, f = s.split(".", 1)
        return f"{_int(i)} {POINT} {_frac(f)}"
    return _int(s)

def _money(s, sym):                # 45.6$ -> چل و پێنج دولار و شەش سەنت
    main, cent = CURR[sym]; s = s.translate(_ASCII)
    i, _, f = s.partition(".")
    out = f"{_int(i)} {main}"
    if f.strip("0"): out += f"{AND}{_int(f.lstrip('0'))} {cent}"
    return out

def _time(m):                      # 21:45:30 -> دەمژمێر بیست و ئێک و چل و پێنج خولەک و سیه چرکە
    h, mi, se = m.group(1), m.group(2), m.group(3)
    out = [f"{HOUR} {_int(h)}"]
    if int(mi.translate(_ASCII)): out.append(f"{_int(mi)} {MINUTE}")
    if se and int(se.translate(_ASCII)): out.append(f"{_int(se)} {SECOND}")
    return AND.join(out)

def _date(m):
    g = m.groups()
    if len(g[0].translate(_ASCII)) == 4: y, mo, d = g[0], g[1], g[2]     # 2026/11/23
    else:                                d, mo, y = g[0], g[1], g[2]     # 23/11/2026
    before = m.string[:m.start()].rstrip()
    day = "" if re.search(r"[ڕر]ۆژا$", before) else DAY + " "     # text already says "ڕۆژا" -> don't repeat it
    return f"{day}{_int(d)}، {MONTH} {_int(mo)}، {YEAR} {_int(y)}"

def digits_to_words(text):
    # 0) Arabic decimal/thousands marks -> ASCII; minus; plus
    text = text.replace("٫", ".").replace("٬", ",")
    text = re.sub(rf"(?<!\w)[-−](?=[{D}])", f"{MINUS} ", text)          # -234 / -56.23 (not 2020-2024)
    text = re.sub(rf"(?<=[{D}])\s*\+\s*(?=[{D}])", f" {PLUS} ", text)       # 234+ 78
    # 1) dates  2026/11/23, 23/11/2026 or 10\7\2018 (backslash form used in your data)
    text = re.sub(rf"(?<![{D}])([{D}]{{4}}|[{D}]{{1,2}})[/\\]([{D}]{{1,2}})[/\\]([{D}]{{4}}|[{D}]{{1,2}})(?![{D}])", _date, text)
    # 2) time  21:45:30 / 21:45
    text = re.sub(rf"(?<![{D}])([{D}]{{1,2}}):([{D}]{{2}})(?::([{D}]{{2}}))?(?![{D}])", _time, text)
    # 3) thousands commas  1,250 -> 1250
    text = re.sub(rf"(?<=[{D}]),(?=[{D}]{{3}}(?![{D}]))", "", text)
    # 4) currency  45.6$  /  $45.6
    sym = "".join(re.escape(k) for k in CURR)
    text = re.sub(rf"([{D}]+(?:\.[{D}]+)?)\s*([{sym}])", lambda m: _money(m.group(1), m.group(2)), text)
    text = re.sub(rf"([{sym}])\s*([{D}]+(?:\.[{D}]+)?)", lambda m: _money(m.group(2), m.group(1)), text)
    # 5) decimals and plain numbers
    return re.sub(rf"[{D}]+(?:\.[{D}]+)?", lambda m: _num(m.group(0)), text)

## 2 · Helpers

In [4]:
def read_tsv(tsv, audio_dir):
    rows, miss, empty = [], 0, 0
    with open(tsv, encoding="utf-8-sig") as f:
        lines = f.readlines()
    if HAS_HEADER: lines = lines[1:]
    for l in lines:
        l = l.rstrip("\n")
        if not l.strip(): continue
        p = l.split("\t")
        if len(p) <= max(SPK_COL, PATH_COL, TEXT_COL): continue
        spk, fn, text = p[SPK_COL].strip(), p[PATH_COL].strip(), p[TEXT_COL].strip()
        if not text: empty += 1; continue
        wav = f"{audio_dir}/{fn}"
        if not os.path.exists(wav): miss += 1; continue
        rows.append((spk, fn, text, wav))
    print(f"  {os.path.basename(tsv)}: {len(rows)} usable | {miss} missing-audio | {empty} empty-text")
    return rows

def uid(fn): return os.path.splitext(os.path.basename(fn))[0]

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (x != y)))
        prev = cur
    return prev[-1]

_PUNCT = re.compile(r"[\u060c\u061b\u061f\u066b\u066c.,;:!?\"'()\[\]{}\-\u2013\u2014\u2026]")
def norm_text(t):
    t = _PUNCT.sub(" ", t).replace("\u064a", "\u06cc").replace("\u0643", "\u06a9")
    return re.sub(r"\s+", " ", t).strip()

## 3 · Load the test set + a clean prompt per speaker (from TRAIN data, never a test clip)

In [5]:
train_rows = read_tsv(TRAIN_TSV, AUDIO_DIR)
test_rows  = read_tsv(TEST_TSV,  AUDIO_DIR)
claims = collections.defaultdict(set)
for spk, fn, _, _ in train_rows + test_rows: claims[fn].add(spk)
bad = {fn for fn, s in claims.items() if len(s) > 1}
test = {uid(fn): (wav, text, spk) for spk, fn, text, wav in test_rows if fn not in bad}
train = {uid(fn): (wav, text, spk) for spk, fn, text, wav in train_rows if fn not in bad and uid(fn) not in test}
SPEAKERS = sorted({v[2] for v in train.values()})
per_spk = collections.defaultdict(list)
for u in sorted(test):
    if test[u][2] in SPEAKERS: per_spk[test[u][2]].append(u)
if LIMIT: per_spk = {s: ids[:LIMIT] for s, ids in per_spk.items()}
print("test sentences per speaker:", {s: len(i) for s, i in per_spk.items()})

PROMPT = {}
for s in SPEAKERS:
    for u in sorted(train):
        w, t, sp = train[u]
        if sp != s: continue
        try:
            if 3.0 <= sf.info(w).duration <= 12.0: PROMPT[s] = (w, t); break
        except Exception: continue
assert set(PROMPT) == set(SPEAKERS), "no 3-12 s prompt clip found for some speaker"
print("prompts:", {s: os.path.basename(p[0]) for s, p in PROMPT.items()})

  train_all.tsv: 10945 usable | 0 missing-audio | 0 empty-text
  test_all.tsv: 303 usable | 0 missing-audio | 0 empty-text
test sentences per speaker: {'bafreen': 101, 'abdullah': 101, 'hayfa': 101}
prompts: {'abdullah': '2_44100_a1.wav', 'bafreen': '1_44100_a100.wav', 'hayfa': 'hayfa98_44100_a1.wav'}


## 4 · Synthesize ALL test sentences (resumable, timed)

In [6]:
from cosyvoice.cli.cosyvoice import AutoModel
cosy = AutoModel(model_dir=MODEL_DIR); sr = cosy.sample_rate
timing_path = f"{OUT_DIR}/timing.json"
timing = json.load(open(timing_path)) if os.path.exists(timing_path) else {}
for s, ids in per_spk.items():
    os.makedirs(f"{SYNTH_DIR}/{s}", exist_ok=True); pw, pt = PROMPT[s]
    for k, u in enumerate(ids, 1):
        out = f"{SYNTH_DIR}/{s}/{u}.wav"
        if os.path.exists(out): continue
        text = digits_to_words(test[u][1])
        try:
            if dev == "cuda": torch.cuda.synchronize()
            t0 = time.time()
            chunks = [j["tts_speech"] for j in cosy.inference_zero_shot(
                text, INSTRUCT + pt, pw, stream=False, text_frontend=False)]
            if dev == "cuda": torch.cuda.synchronize()
            el = time.time() - t0
            wav = chunks[0] if len(chunks) == 1 else torch.cat(chunks, dim=1)
            torchaudio.save(out, wav, sr)
            timing[f"{s}/{u}"] = dict(sec=el, audio_sec=wav.shape[1] / sr)
        except Exception as e:
            print(f"  !! synth failed {s}/{u}: {e}")
        if k % 25 == 0 or k == len(ids):
            print(f"  {s}: {k}/{len(ids)}"); json.dump(timing, open(timing_path, "w"))
json.dump(timing, open(timing_path, "w"))
del cosy
if dev == "cuda": torch.cuda.empty_cache()

/workspace/envs/cosyvoice/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-10-13 08:34:26,473] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/workspace/envs/cosyvoice/bin/x86_64-conda-linux-gnu-ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/workspace/envs/cosyvoice/bin/x86_64-conda-linux-gnu-ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
/workspace/envs/cosyvoice/lib/python3.10/site-packages/lightning/fabric/__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
2026-10-13 08:35:32,045 INFO input frame rate=25
/workspace/envs/cosyvoice/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, nam

2026-10-13 08:35:42,943 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/revisions HTTP/1.1" 403 137
2026-10-13 08:35:42,944 - modelscope - ERROR - Authentication token does not exist, 
2026-10-13 08:35:42,945 INFO no frontend is avaliable
  0%|          | 0/1 [00:00<?, ?it/s]/workspace/envs/cosyvoice/lib/python3.10/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
2026-10-13 08:35:55,268 INFO synthesis text لەوما خو ئێخستە سەر پشتا خو و بەرێ خو دا تایێت مشمشێ و دەڤێ خو ڤەکر، کا چ ناکەڤنە د دەڤێ وی دا.
/workspace/CosyV

  bafreen: 25/101


...

  hayfa: 75/101


  0%|          | 0/1 [00:00<?, ?it/s]2026-10-13 08:58:55,081 INFO synthesis text ئەرێ مەبەستا وی ئەوە کو هێزێن وی ل چیایێن قەندیلی نە ل باشووری کوردستانێ؟
2026-10-13 08:58:58,939 INFO yield speech len 6.08, rtf 0.6346255932983599
  0%|          | 0/1 [00:00<?, ?it/s]2026-10-13 08:58:58,971 WARNING synthesis text ئەرێ موسلمانە یێ گوهداریا پێغەمبەری سلاڤێت خودێ ل سەر بن نەکەت؟ too short than prompt text You are a helpful assistant.<|endofprompt|>ڕەنگ ل وی نەما دەمێ وی دیتی کانێ چاوا وی داعشی سەرێ ھەڤالێ وی بڕی و ڤزکا خوینا سور ب سەر وارا ڤە چوو., this may lead to bad performance
2026-10-13 08:59:00,320 INFO synthesis text ئەرێ موسلمانە یێ گوهداریا پێغەمبەری سلاڤێت خودێ ل سەر بن نەکەت؟
2026-10-13 08:59:03,162 INFO yield speech len 4.28, rtf 0.6640959008831844
  0%|          | 0/1 [00:00<?, ?it/s]2026-10-13 08:59:04,578 INFO synthesis text ئەرێ میدیایا مە ل دەڤەرا بەهدینان بوویە میدیایەکا پیشەیی، مۆدێرن و سەردەم؟
2026-10-13 08:59:08,169 INFO yield speech len 5.52, rtf 0.650473738062209
  0

  hayfa: 100/101


  0%|          | 0/1 [00:00<?, ?it/s]2026-10-13 09:00:45,144 WARNING synthesis text ئەرێکرن، نیشانکرن، شیرانی، خەلاتبرین کرنە یەک. too short than prompt text You are a helpful assistant.<|endofprompt|>ڕەنگ ل وی نەما دەمێ وی دیتی کانێ چاوا وی داعشی سەرێ ھەڤالێ وی بڕی و ڤزکا خوینا سور ب سەر وارا ڤە چوو., this may lead to bad performance
2026-10-13 09:00:46,520 INFO synthesis text ئەرێکرن، نیشانکرن، شیرانی، خەلاتبرین کرنە یەک.
2026-10-13 09:00:49,430 INFO yield speech len 4.28, rtf 0.6800642080396135
100%|██████████| 1/1 [00:04<00:00,  4.29s/it]


  hayfa: 101/101


## 5 · Scorers (ECAPA, UTMOS, optional ASR)

In [9]:
from speechbrain.inference.speaker import EncoderClassifier
enc = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb",
        savedir="/workspace/tmp/ecapa", run_opts={"device": dev})
ums = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True).to(dev).eval()
def load16(p):
    w, s_ = torchaudio.load(p); w = w.mean(0, keepdim=True)
    return torchaudio.functional.resample(w, s_, 16000) if s_ != 16000 else w
def emb(p):
    with torch.no_grad(): e = enc.encode_batch(load16(p).to(dev)).squeeze()
    return torch.nn.functional.normalize(e, dim=0).cpu().numpy()
def utmos(p):
    with torch.no_grad(): return float(ums(load16(p).to(dev), 16000))

asr = None
if ASR_MODEL:
    from transformers import pipeline
    asr = pipeline("automatic-speech-recognition", model=ASR_MODEL, device=0 if dev == "cuda" else -1)
    def transcribe(p):
        a = {"raw": load16(p).squeeze(0).numpy(), "sampling_rate": 16000}
        return asr(a, generate_kwargs={"language": ASR_LANG})["text"] if ASR_LANG else asr(a)["text"]

2026-10-13 09:10:05,173 INFO Fetch hyperparams.yaml: Using symlink found at '/workspace/tmp/ecapa/hyperparams.yaml'
2026-10-13 09:10:05,426 INFO Fetch embedding_model.ckpt: Using symlink found at '/workspace/tmp/ecapa/embedding_model.ckpt'
2026-10-13 09:10:05,435 INFO Fetch mean_var_norm_emb.ckpt: Using symlink found at '/workspace/tmp/ecapa/mean_var_norm_emb.ckpt'
2026-10-13 09:10:05,445 INFO Fetch classifier.ckpt: Using symlink found at '/workspace/tmp/ecapa/classifier.ckpt'
2026-10-13 09:10:05,454 INFO Fetch label_encoder.txt: Using symlink found at '/workspace/tmp/ecapa/label_encoder.ckpt'
2026-10-13 09:10:05,455 INFO Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
2026-10-13 09:10:06,543 WARNING Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.
Using cache found in /root/.cache/torch/hub/tarepan_SpeechMOS_v1.2.0


## 6 · Score every utterance

In [10]:
rows = []
for s, ids in per_spk.items():
    pref = emb(PROMPT[s][0])
    for i, u in enumerate(ids, 1):
        sp = f"{SYNTH_DIR}/{s}/{u}.wav"
        if not os.path.exists(sp): continue
        real = test[u][0]; e = emb(sp)
        sd = sf.info(sp).duration; rd = sf.info(real).duration; tm = timing.get(f"{s}/{u}")
        r = dict(speaker=s, utt=u, text=test[u][1],
                 simo_prompt=float(np.dot(pref, e)), simo_real=float(np.dot(emb(real), e)),
                 utmos=utmos(sp), synth_sec=sd, real_sec=rd, dur_ratio=sd / rd if rd else float("nan"),
                 rtf=(tm["sec"] / tm["audio_sec"]) if tm and tm["audio_sec"] else float("nan"))
        if asr:
            hyp = transcribe(sp); ref = norm_text(digits_to_words(test[u][1])); hn = norm_text(hyp)
            rw, hw = ref.split(), hn.split()
            r.update(asr_text=hyp, wer=edit_distance(rw, hw) / max(len(rw), 1), cer=edit_distance(ref, hn) / max(len(ref), 1))
        rows.append(r)
        if i % 25 == 0 or i == len(ids): print(f"  scored {s}: {i}/{len(ids)}")

/workspace/envs/cosyvoice/lib/python3.10/site-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  s = torchaudio.io.StreamReader(src, format, None, buffer_size)


  scored bafreen: 25/101
  scored bafreen: 50/101
  scored bafreen: 75/101
  scored bafreen: 100/101
  scored bafreen: 101/101
  scored abdullah: 25/101
  scored abdullah: 50/101
  scored abdullah: 75/101
  scored abdullah: 100/101
  scored abdullah: 101/101
  scored hayfa: 25/101
  scored hayfa: 50/101
  scored hayfa: 75/101
  scored hayfa: 100/101
  scored hayfa: 101/101


## 7 · Results table, files, worst sentences

In [11]:
import pandas as pd
df = pd.DataFrame(rows)
df.to_csv(f"{OUT_DIR}/per_utterance.csv", index=False, encoding="utf-8-sig")
metrics = ["simo_prompt", "simo_real", "utmos", "rtf", "dur_ratio"] + (["cer", "wer"] if asr else [])
df["dur_outlier"] = ~df.dur_ratio.between(0.5, 2.0)      # likely truncated / looping
by_spk = df.groupby("speaker")[metrics + ["dur_outlier"]].agg({**{m: "mean" for m in metrics}, "dur_outlier": "sum"})
by_spk.insert(0, "n", df.groupby("speaker").size())
overall = pd.DataFrame({**{m: [df[m].mean()] for m in metrics}, "dur_outlier": [df.dur_outlier.sum()], "n": [len(df)]}, index=["ALL"])
summary = pd.concat([by_spk, overall[by_spk.columns]])
summary.to_csv(f"{OUT_DIR}/summary.csv", encoding="utf-8-sig")
json.dump(summary.to_dict("index"), open(f"{OUT_DIR}/summary.json", "w"), indent=1, ensure_ascii=False)
print("SIM-o / UTMOS: higher is better | RTF, CER, WER: lower is better | dur_ratio ~1.0 expected")
display(summary.round(3))
print("lowest SIM-o sentences (listen to these):")
display(df.nsmallest(5, "simo_real")[["speaker", "utt", "simo_real", "utmos", "dur_ratio"]])

SIM-o / UTMOS: higher is better | RTF, CER, WER: lower is better | dur_ratio ~1.0 expected


,n,simo_prompt,simo_real,utmos,rtf,dur_ratio,dur_outlier
abdullah,101,0.798,0.502,3.536,0.895,1.059,0
bafreen,101,0.699,0.606,3.766,1.285,1.008,4
hayfa,101,0.755,0.600,3.559,0.954,1.041,0
ALL,303,0.750,0.569,3.621,1.045,1.036,4


lowest SIM-o sentences (listen to these):


,speaker,utt,simo_real,utmos,dur_ratio
4,bafreen,1_44100_a9904,-0.031633,2.008604,0.072213
28,bafreen,1_44100_a9928,-0.027880,2.922091,0.050982
1,bafreen,1_44100_a9901,0.017666,2.892092,0.041678
15,bafreen,1_44100_a9915,0.113140,3.023997,0.044219
136,abdullah,2_44100_a7402,0.275168,3.817994,1.174564


In [14]:
for _, r in bad.iterrows(): os.remove(f"{SYNTH_DIR}/{r.speaker}/{r.utt}.wav")

AttributeError: 'set' object has no attribute 'iterrows'

In [15]:
fails = df[~df.dur_ratio.between(0.5, 2.0)]
display(fails[["speaker","utt","real_sec","synth_sec","dur_ratio","simo_real","utmos","text"]])

for _, r in fails.iterrows():
    os.remove(f"{SYNTH_DIR}/{r.speaker}/{r.utt}.wav")
print("deleted", len(fails), "wavs")

,speaker,utt,real_sec,synth_sec,dur_ratio,simo_real,utmos,text
1,bafreen,1_44100_a9901,2.879184,0.12,0.041678,0.017666,2.892092,بلا ئەزبەنی، جارێ دا شیڤا ژی بدەینێ.
4,bafreen,1_44100_a9904,5.539184,0.40,0.072213,-0.031633,2.008604,پشتی سێزدە سالان هێلا ئاسمانیا تورکیا سەر ژ نو...
15,bafreen,1_44100_a9915,1.809184,0.08,0.044219,0.113140,3.023997,ساندەرسێ روباری؟
28,bafreen,1_44100_a9928,1.569184,0.08,0.050982,-0.027880,2.922091,ژ بابێت مەنە.


deleted 4 wavs
